# 02 — Лаборатория волатильности: HV против IV, скью и временная структура

Вы:
1. Посчитаете **историческую волатильность** по синтетическому ряду цен (с фиксированным seed).
2. Сделаете круг «туда-обратно» с **подразумеваемой волатильностью** через `pricing.implied_vol` /
   `pricing.bsm_price`.
3. Построите **скью** (IV против страйка) и **временную структуру** (IV против DTE) по тестовым
   доскам.
4. Сравните режимы волатильности **DEMO**, **LOWVOL** и **HIGHVOL**.

Всё оффлайн. Спот DEMO — **$100**, IV ~**25%**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import pricing, analyzer, data

## 1. Историческая волатильность по синтетическому ряду цен

Смоделируем геометрическое случайное блуждание с фиксированным seed и известной «истинной»
волатильностью, а затем оценим HV обратно по дневным логарифмическим доходностям:
`std(log returns) * sqrt(252)`. Оценка должна лечь рядом с входным значением.

In [ ]:
rng = np.random.default_rng(42)
true_vol, days, S0 = 0.25, 252, 100.0
daily = rng.normal(0, true_vol / np.sqrt(252), days)   # дневные логарифмические доходности
prices = S0 * np.exp(np.cumsum(daily))
log_ret = np.diff(np.log(prices))
hv = log_ret.std(ddof=1) * np.sqrt(252)
print(f'входная (истинная) волатильность: {true_vol:.3f}   оценка HV: {hv:.3f}')

HV — это измерение реализованного движения *назад*. Дальше — IV, число *вперёд*, которое рынок
закладывает в опционы.

## 2. Круг «туда-обратно» с подразумеваемой волатильностью

IV — это волатильность, которая воспроизводит рыночную цену. Извлечём её из mid ATM-колла DEMO
(3.91), затем подставим обратно в `bsm_price` и убедимся, что получили 3.91: `implied_vol` и
`bsm_price` взаимно обратны.

In [ ]:
mid = 3.91
iv = pricing.implied_vol('call', price=mid, spot=100, strike=100, t=45/365)
back = pricing.bsm_price('call', 100, 100, 45/365, iv)
print(f'подразумеваемая волатильность: {iv:.4f}')
print(f'переоценка при этой IV: {back:.2f}  (воспроизводит исходные {mid})')

## 3. Скью: IV меняется по страйкам

Загрузим DEMO, возьмём строки на 45 DTE и построим колонку `iv` самой доски против страйка. У акций
видна **пут-скью**: у путов вне денег (низкие страйки) IV выше, чем у коллов вне денег (высокие
страйки).

In [ ]:
demo = data.load_sample_chain('DEMO')
c45 = demo[(demo.kind == 'call') & (demo.expiry_days == 45)].sort_values('strike')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(c45.strike, c45.iv, 'o-')
ax.axvline(100, color='k', ls='--', lw=1, label='спот=100')
ax.set_xlabel('страйк'); ax.set_ylabel('подразумеваемая волатильность'); ax.set_title('Скью DEMO на 45 DTE'); ax.legend()
plt.show()

Кривая идёт **вниз слева направо**: нижние страйки (путы на падение) задраны риском обвала и
спросом на защиту. Именно поэтому продажа премии на стороне путов собирает больше, чем на стороне
коллов.

## 4. Временная структура: ATM IV по экспирациям

Теперь зафиксируем страйк около денег (100) и построим IV против DTE. Восходящий наклон — контанго
(спокойствие); нисходящий — бэквордация (ближний стресс). DEMO в лёгкой бэквордации.

In [ ]:
atm = demo[(demo.kind == 'call') & (demo.strike == 100)].sort_values('expiry_days')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(atm.expiry_days, atm.iv, 's-')
ax.set_xlabel('DTE'); ax.set_ylabel('ATM подразумеваемая волатильность'); ax.set_title('Временная структура ATM у DEMO')
plt.show()
print(atm[['expiry_days', 'iv']].to_string(index=False))

## 5. Сравним три режима волатильности

DEMO (~25%), LOWVOL (~14%), HIGHVOL (~55%). Достанем ATM IV каждой доски, чтобы увидеть, насколько
по-разному выглядит «норма» у разных бумаг, — именно поэтому абсолютное значение IV не значит
ничего.

In [ ]:
def atm_iv(name):
    ch = data.load_sample_chain(name)
    spot = ch.spot.iloc[0]
    calls = ch[ch.kind == 'call'].copy()
    row = calls.iloc[(calls.strike - spot).abs().argmin()]
    return spot, row.strike, row.iv

for nm in ['LOWVOL', 'DEMO', 'HIGHVOL']:
    spot, k, iv = atm_iv(nm)
    print(f'{nm:8s} спот {spot:6.1f}  ATM страйк {k:6.1f}  ATM IV {iv:.3f}')

## 6. Скью бок о бок

Наложим скью трёх досок друг на друга (нормируем ось x на strike/spot, чтобы они совпали). HIGHVOL
сидит далеко вверху, LOWVOL — далеко внизу: форма та же, уровни совершенно разные.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for nm in ['LOWVOL', 'DEMO', 'HIGHVOL']:
    ch = data.load_sample_chain(nm)
    spot = ch.spot.iloc[0]
    dte = sorted(ch.expiry_days.unique())[len(ch.expiry_days.unique())//2]
    sl = ch[(ch.kind == 'call') & (ch.expiry_days == dte)].sort_values('strike')
    ax.plot(sl.strike / spot, sl.iv, 'o-', label=f'{nm} ({dte}д)')
ax.axvline(1.0, color='k', ls='--', lw=1)
ax.set_xlabel('страйк / спот (денежность)'); ax.set_ylabel('IV'); ax.set_title('Скью в разных режимах волатильности'); ax.legend()
plt.show()

## 7. Ожидаемое (подразумеваемое) движение

`analyzer.expected_move(spot, vol, t)` возвращает движение в одну сигму `spot * vol * sqrt(t)` —
вашу линейку для вопроса «а какое движение закладывает рынок?». Сравним подразумеваемое движение
DEMO на 45 DTE при 25% и при событийных 55%.

In [ ]:
em_calm  = analyzer.expected_move(100, 0.25, 45/365)
em_event = analyzer.expected_move(100, 0.55, 45/365)
print(f'движение в 1 сигму при IV 25%: +/- {em_calm:.2f}  (примерно к {100-em_calm:.0f}/{100+em_calm:.0f})')
print(f'движение в 1 сигму при IV 55%: +/- {em_event:.2f}  (примерно к {100-em_event:.0f}/{100+em_event:.0f})')

## Эксперименты

1. В разделе 1 поменяйте seed и `true_vol` (попробуйте 0.10 и 0.60). Отслеживает ли оценка HV
   входное значение? Увеличьте `days` до 1000 — оценка становится точнее?
2. В разделе 2 извлеките IV из mid **пута 95** (1.58) и **колла 110** (0.73). Совпадают ли
   полученные IV по страйкам с колонкой `iv` в доске (это и есть скью)?
3. В разделе 3 постройте скью по **путам** вместо коллов. Форма та же? (Должна быть — в рамках
   одной модели IV является свойством страйка, а не типа опциона.)
4. В разделе 4 постройте временную структуру для **HIGHVOL** (спот 62). Это контанго или
   бэквордация и что это подразумевает о ближнем событийном риске?
5. В разделе 7 подберите, при какой IV движение в одну сигму на 45 DTE достигает 10 пунктов.
   Решайте перебором значений — это и есть то «подразумеваемое движение», которое обязан
   превзойти покупатель стрэддла.